# RL Training Notebook (Google Colab) — Phase 07 v2

This notebook:
1. builds train / held-out splits (held-out never used for adaptive selection);
2. runs base held-out eval (1 rollout / task);
3. preflight → **variance-qualified** tier 2/3 train pool;
4. trains with group-relative PG + frozen-reference KL (skips unscored / low-variance groups);
5. compares β=1e-5 vs β=1e-2 on a **shared schedule**;
6. logs skip reasons, checkpoints every nominal step, audits reward pathways.


In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

assert os.path.exists('/content'), 'Run this notebook in Google Colab'
subprocess.run(['nvidia-smi'], check=True)

REPO_URL = 'https://github.com/prashere/evaluator_gym.git'
try:
    from google.colab import userdata
    github_token = userdata.get('GITHUB_TOKEN')
except Exception:
    github_token = os.environ.get('GITHUB_TOKEN')
if github_token:
    REPO_URL = f'https://{github_token}@github.com/prashere/evaluator_gym.git'

REPO = Path('/content/evaluator_gym')
if not REPO.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
subprocess.run(['git', 'pull', '--ff-only'], check=True)

# Bootstrap from training/ on disk before pip import (clone ≠ installed package).
_bootstrap_file = REPO / 'src/evaluator_gym/training/colab_bootstrap.py'
_spec = importlib.util.spec_from_file_location('colab_bootstrap', _bootstrap_file)
_bootstrap = importlib.util.module_from_spec(_spec)
assert _spec.loader is not None
_spec.loader.exec_module(_bootstrap)
SRC = _bootstrap.install_repo_src(REPO)
print('Added to sys.path:', SRC)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO)], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'transformers==4.57.6', 'peft==0.17.1', 'bitsandbytes==0.47.0',
    'accelerate==1.10.1', 'mlflow==3.10.0', 'matplotlib>=3.8,<4', 'tqdm>=4.66,<5',
], check=True)

print('Phase 07 imports:', _bootstrap.verify_imports())

from google.colab import drive
drive.mount('/content/drive')

from evaluator_gym.training.phase07_core import DEFAULT_OUTPUT_ROOT, ensure_output_root

OUTPUT_ROOT = ensure_output_root(DEFAULT_OUTPUT_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
import json
import sys
from collections import Counter
from pathlib import Path

REPO = Path('/content/evaluator_gym')
_src = str((REPO / 'src').resolve())
if _src not in sys.path:
    sys.path.insert(0, _src)

from evaluator_gym.training.colab_bootstrap import install_repo_src

install_repo_src(REPO)

from evaluator_gym.training.phase07_core import build_phase07_splits

TRAIN_CONFIG, HELDOUT_POOL_CONFIG, TRAIN_TASK_ROWS, HELDOUT_TASK_ROWS = build_phase07_splits()
TRAIN_TASKS = [row.as_dict() for row in TRAIN_TASK_ROWS]
HELDOUT_TASKS = [row.as_dict() for row in HELDOUT_TASK_ROWS]

print('Train:', TRAIN_CONFIG.to_dict())
print('Held-out pool:', HELDOUT_POOL_CONFIG.to_dict())
print('Tier counts:', dict(Counter(row['tier'] for row in TRAIN_TASKS)))
print('Held-out IDs:', [row['task_id'] for row in HELDOUT_TASKS])

In [ ]:
import random
import sys
from pathlib import Path
from statistics import fmean

REPO = Path('/content/evaluator_gym')
_src = str((REPO / 'src').resolve())
if _src not in sys.path:
    sys.path.insert(0, _src)

from evaluator_gym.training.colab_bootstrap import install_repo_src

install_repo_src(REPO)

import mlflow
import numpy as np
import torch
from evaluator_gym.rubric import RUBRIC_VERSION
from evaluator_gym.training.phase07_core import (
    BETAS,
    DEFAULT_OUTPUT_ROOT,
    GROUP_SIZE,
    HELDOUT_ROLLOUTS,
    MAX_COMPLETION_TOKENS,
    MAX_RETRIES,
    MIN_REWARD_STD,
    PREFLIGHT_PROBES_PER_TASK,
    RUN_SEED,
    SMOKE_STEPS,
    TRAIN_STEPS,
    build_training_schedule,
    build_training_metric,
    decide_training_step,
    ensure_output_root,
    should_save_checkpoint,
    summarize_evaluation,
    summarize_preflight,
    summarize_rejections,
)
from evaluator_gym.training.phase07_runtime import (
    append_jsonl,
    exploit_search,
    read_jsonl,
    run_async,
    score_text,
    strict_json,
)
from evaluator_gym.training.phase07_live import LiveRunLogger, tqdm_progress

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
MODEL_REVISION = '7ae557604adf67be50417f59c2c2f167def9a775'

OUTPUT_ROOT = ensure_output_root(globals().get('OUTPUT_ROOT', DEFAULT_OUTPUT_ROOT))

assert RUBRIC_VERSION == '0.1.2', f'Expected rubric 0.1.2, got {RUBRIC_VERSION}'
assert torch.cuda.is_available(), 'A Colab GPU is required'
print('GPU:', torch.cuda.get_device_name(0))
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('MAX_COMPLETION_TOKENS:', MAX_COMPLETION_TOKENS, '| MIN_REWARD_STD:', MIN_REWARD_STD)

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed


def build_policy():
    set_seed(RUN_SEED)
    quantization = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
    tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
        quantization_config=quantization,
        device_map={'': 0},
        torch_dtype=torch.float16,
    )
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model = get_peft_model(model, LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias='none',
        task_type='CAUSAL_LM',
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    ))
    unexpected = [name for name, p in model.named_parameters() if p.requires_grad and 'lora_' not in name]
    assert not unexpected, f'Frozen-reference invariant failed: {unexpected[:5]}'
    return model, tokenizer


def render_prompt(task, tokenizer):
    text = tokenizer.apply_chat_template(task['prompt'], tokenize=False, add_generation_prompt=True)
    encoded = tokenizer(text, return_tensors='pt').to('cuda')
    assert encoded['input_ids'].shape[1] + MAX_COMPLETION_TOKENS <= 32768
    return encoded


def generate_valid_group(model, tokenizer, task, nominal_step, rejection_path, live_logger=None):
    encoded = render_prompt(task, tokenizer)
    prompt_length = encoded['input_ids'].shape[1]
    samples = []
    model.eval()
    slot_iter = tqdm_progress(
        range(GROUP_SIZE),
        desc=f'group {task["task_id"]} s{nominal_step}',
        total=GROUP_SIZE,
        leave=False,
    )
    for group_index in slot_iter:
        accepted = None
        for retry in range(MAX_RETRIES):
            sample_seed = RUN_SEED + nominal_step * 1000 + group_index * 10 + retry
            torch.manual_seed(sample_seed)
            with torch.no_grad():
                output = model.generate(
                    **encoded,
                    do_sample=True,
                    temperature=0.8,
                    max_new_tokens=MAX_COMPLETION_TOKENS,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )
            completion_ids = output[0, prompt_length:].detach()
            text = tokenizer.decode(completion_ids, skip_special_tokens=True)
            reward, parse_result, audit = run_async(score_text(task, text))
            if live_logger is not None:
                live_logger.log_group_slot(
                    nominal_step=nominal_step,
                    task_id=task['task_id'],
                    group_index=group_index,
                    retry=retry,
                    scored=reward is not None,
                    reward=reward,
                    error_class=parse_result.get('error_class'),
                )
            candidate = {
                'task_id': task['task_id'], 'tier': task['tier'], 'nominal_step': nominal_step,
                'group_index': group_index, 'retry': retry, 'seed': sample_seed,
                'completion': text, 'completion_tokens': int(completion_ids.numel()),
                'reward': reward, 'parse_result': parse_result, 'reward_audit': audit,
            }
            if reward is None:
                append_jsonl(rejection_path, candidate)
                slot_iter.set_postfix(slot=group_index + 1, retry=retry, status='reject')
                continue
            candidate['prompt_ids'] = encoded['input_ids'][0].detach()
            candidate['completion_ids'] = completion_ids
            accepted = candidate
            slot_iter.set_postfix(slot=group_index + 1, reward=f'{reward:.2f}')
            break
        if accepted is None:
            model.train()
            return None
        samples.append(accepted)
    model.train()
    return samples


def token_statistics(model, sample):
    tokens = torch.cat([sample['prompt_ids'], sample['completion_ids']]).unsqueeze(0)
    targets = sample['completion_ids']
    keep = targets.numel() + 1
    policy_logits = model(tokens, logits_to_keep=keep).logits[0, :-1].float()
    policy_log_probs = policy_logits.log_softmax(-1)
    policy_token_logp = policy_log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
    entropy = -(policy_log_probs.exp() * policy_log_probs).sum(-1).mean()
    with torch.no_grad(), model.disable_adapter():
        reference_logits = model(tokens, logits_to_keep=keep).logits[0, :-1].float()
        reference_token_logp = reference_logits.log_softmax(-1).gather(1, targets.unsqueeze(1)).squeeze(1)
    log_ratio = reference_token_logp - policy_token_logp
    k3 = (torch.exp(log_ratio) - log_ratio - 1).mean()
    return policy_token_logp.mean(), k3, entropy

In [ ]:
def evaluate_policy(model, tokenizer, run_name):
    run_dir = OUTPUT_ROOT / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    transcript_path = run_dir / 'heldout_rollouts.jsonl'
    if transcript_path.exists():
        transcript_path.unlink()
    live = LiveRunLogger(run_name, run_dir)
    live.print_banner(f'Held-out evaluation — {run_name} ({len(HELDOUT_TASKS)} tasks × {HELDOUT_ROLLOUTS} rollout)')
    model.eval()
    total = len(HELDOUT_TASKS) * HELDOUT_ROLLOUTS
    rollout_index_global = 0
    progress = tqdm_progress(range(total), desc=f'held-out {run_name}', total=total)
    for task_index, task in enumerate(HELDOUT_TASKS):
        encoded = render_prompt(task, tokenizer)
        prompt_length = encoded['input_ids'].shape[1]
        for rollout_index in range(HELDOUT_ROLLOUTS):
            seed = RUN_SEED + task_index * HELDOUT_ROLLOUTS + rollout_index
            torch.manual_seed(seed)
            with torch.no_grad():
                output = model.generate(
                    **encoded, do_sample=True, temperature=0.8,
                    max_new_tokens=MAX_COMPLETION_TOKENS,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )
            ids = output[0, prompt_length:]
            text = tokenizer.decode(ids, skip_special_tokens=True)
            reward, parse_result, audit = run_async(score_text(task, text))
            rollout_index_global += 1
            append_jsonl(transcript_path, {
                'run': run_name, 'task_id': task['task_id'], 'tier': task['tier'],
                'rollout_index': rollout_index, 'seed': seed, 'completion': text,
                'completion_tokens': int(ids.numel()), 'reward': reward,
                'parse_result': parse_result, 'reward_audit': audit,
            })
            live.log_eval_rollout(
                index=rollout_index_global,
                total=total,
                task_id=task['task_id'],
                tier=task['tier'],
                reward=reward,
                completion_tokens=int(ids.numel()),
                parse_result=parse_result,
            )
            progress.update(1)
            parse_rate = live.eval_scored / live.eval_total if live.eval_total else 0.0
            progress.set_postfix(
                task=task['task_id'],
                tier=task['tier'],
                parse=f'{parse_rate:.0%}',
                reward=f'{reward:.2f}' if reward is not None else '—',
            )
            if rollout_index_global % 5 == 0 or rollout_index_global == total:
                live.refresh_panel()
    progress.close()
    rows = read_jsonl(transcript_path)
    summary = summarize_evaluation(rows)
    (run_dir / 'heldout_summary.json').write_text(json.dumps(summary, indent=2))
    live.emit(f'Held-out summary ({run_name}): ' + json.dumps(summary))
    live.refresh_panel()
    return summary


base_model, base_tokenizer = build_policy()
BASE_HELDOUT = evaluate_policy(base_model, base_tokenizer, 'base')
print(json.dumps(BASE_HELDOUT, indent=2))
del base_model
torch.cuda.empty_cache()

In [ ]:
from peft import get_peft_model_state_dict, set_peft_model_state_dict


def save_checkpoint(run_dir, nominal_step, model, optimizer, optimizer_applied_steps):
    checkpoint = run_dir / f'checkpoint-{nominal_step}'
    checkpoint.mkdir(parents=True, exist_ok=True)
    torch.save({
        'nominal_step': nominal_step,
        'optimizer_applied_steps': optimizer_applied_steps,
        'adapter': get_peft_model_state_dict(model),
        'optimizer': optimizer.state_dict(),
        'python_rng': random.getstate(),
        'numpy_rng': np.random.get_state(),
        'torch_rng': torch.get_rng_state(),
        'cuda_rng': torch.cuda.get_rng_state_all(),
    }, checkpoint / 'state.pt')
    return checkpoint


def run_preflight(model, tokenizer):
    probes = []
    live = LiveRunLogger('preflight', OUTPUT_ROOT)
    live.print_banner(f'Preflight — {len(TRAIN_TASK_ROWS)} tasks × {PREFLIGHT_PROBES_PER_TASK} probes')
    model.eval()
    task_progress = tqdm_progress(
        list(enumerate(TRAIN_TASK_ROWS)),
        desc='preflight tasks',
        total=len(TRAIN_TASK_ROWS),
    )
    for task_index, task_row in task_progress:
        task = task_row.as_dict()
        encoded = render_prompt(task, tokenizer)
        prompt_length = encoded['input_ids'].shape[1]
        task_probes = []
        for probe_index in range(PREFLIGHT_PROBES_PER_TASK):
            seed = RUN_SEED + 500000 + task_index * 100 + probe_index
            torch.manual_seed(seed)
            with torch.no_grad():
                output = model.generate(
                    **encoded, do_sample=True, temperature=0.8,
                    max_new_tokens=MAX_COMPLETION_TOKENS,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )
            ids = output[0, prompt_length:]
            text = tokenizer.decode(ids, skip_special_tokens=True)
            reward, parse_result, _ = run_async(score_text(task, text))
            row = {
                'task_id': task['task_id'], 'tier': task['tier'], 'probe_index': probe_index,
                'reward': reward, 'parse_result': parse_result,
                'completion_tokens': int(ids.numel()),
            }
            probes.append(row)
            task_probes.append(row)
        live.log_preflight_task(
            task_index=task_index,
            total_tasks=len(TRAIN_TASK_ROWS),
            task_id=task['task_id'],
            tier=task['tier'],
            probes=task_probes,
        )
        scored = sum(item.get('reward') is not None for item in task_probes)
        task_progress.set_postfix(task=task['task_id'], parse=f'{scored}/{PREFLIGHT_PROBES_PER_TASK}')
        if (task_index + 1) % 5 == 0:
            live.refresh_panel()
    task_progress.close()
    model.train()
    live.refresh_panel()
    return summarize_preflight(probes)


def train_run(beta, run_name, total_steps, trainable_ids, training_schedule, resume_checkpoint=None):
    assert beta > 0
    run_dir = OUTPUT_ROOT / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = run_dir / 'metrics.jsonl'
    rollout_path = run_dir / 'training_rollouts.jsonl'
    rejection_path = run_dir / 'rejected_unscored.jsonl'
    config_path = run_dir / 'config.json'
    run_config = {
        'model': MODEL_ID, 'model_revision': MODEL_REVISION, 'rubric_version': RUBRIC_VERSION,
        'beta': beta, 'seed': RUN_SEED, 'total_steps': total_steps, 'group_size': GROUP_SIZE,
        'max_retries': MAX_RETRIES, 'max_completion_tokens': MAX_COMPLETION_TOKENS,
        'min_reward_std': MIN_REWARD_STD,
        'selection_method': 'adaptive_variance_qualified',
        'trainable_task_ids': sorted(trainable_ids),
        'schedule_task_ids': [row.task_id if row else None for row in training_schedule[:total_steps]],
        'train_generator': TRAIN_CONFIG.to_dict(),
        'heldout_pool_generator': HELDOUT_POOL_CONFIG.to_dict(),
    }
    model, tokenizer = build_policy()
    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=1e-5)
    start_step = 0
    optimizer_applied_steps = 0
    if resume_checkpoint:
        state = torch.load(Path(resume_checkpoint) / 'state.pt', map_location='cpu', weights_only=False)
        set_peft_model_state_dict(model, state['adapter'])
        optimizer.load_state_dict(state['optimizer'])
        random.setstate(state['python_rng'])
        np.random.set_state(state['numpy_rng'])
        torch.set_rng_state(state['torch_rng'])
        torch.cuda.set_rng_state_all(state['cuda_rng'])
        start_step = state['nominal_step']
        optimizer_applied_steps = state.get('optimizer_applied_steps', 0)
    elif metrics_path.exists() or rollout_path.exists():
        raise RuntimeError(f'{run_dir} already contains a run; resume it or use another run name')
    config_path.write_text(json.dumps(run_config, indent=2))

    mlflow.set_tracking_uri((OUTPUT_ROOT / 'mlruns').as_uri())
    mlflow.set_experiment('phase07-online-rl')
    skip_counts = Counter()
    live = LiveRunLogger(run_name, run_dir)
    live.print_banner(f'Training — {run_name} (β={beta:g}, {total_steps} nominal steps)')
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({
            'model': MODEL_ID, 'model_revision': MODEL_REVISION,
            'rubric_version': RUBRIC_VERSION, 'beta': beta,
            'train_seed': TRAIN_CONFIG.seed, 'heldout_pool_seed': HELDOUT_POOL_CONFIG.seed,
            'group_size': GROUP_SIZE, 'max_retries': MAX_RETRIES,
            'trainable_tasks': len(trainable_ids),
        })
        progress = tqdm_progress(
            range(start_step, total_steps),
            desc=run_name,
            total=total_steps,
            initial=start_step,
        )
        for step in progress:
            nominal_step = step + 1
            task_row = training_schedule[step] if step < len(training_schedule) else None
            if task_row is None:
                metric = build_training_metric(
                    nominal_step=nominal_step, task=None, rewards=None, kl=None, entropy=None,
                    mean_completion_length=None, optimizer_applied=False, skip_reason='no_trainable_task',
                )
                append_jsonl(metrics_path, metric)
                skip_counts['no_trainable_task'] += 1
                live.log_training_step(metric, beta=beta)
                if should_save_checkpoint(step, total_steps):
                    save_checkpoint(run_dir, nominal_step, model, optimizer, optimizer_applied_steps)
                continue
            task = task_row.as_dict()
            samples = generate_valid_group(
                model, tokenizer, task, nominal_step, rejection_path, live_logger=live,
            )
            if samples is None:
                metric = build_training_metric(
                    nominal_step=nominal_step, task=task_row, rewards=None, kl=None, entropy=None,
                    mean_completion_length=None, optimizer_applied=False, skip_reason='incomplete_group',
                )
                metric['rejection_summary'] = summarize_rejections(
                    [row for row in read_jsonl(rejection_path) if row.get('nominal_step') == nominal_step]
                )
                append_jsonl(metrics_path, metric)
                skip_counts['incomplete_group'] += 1
                live.log_training_step(metric, beta=beta)
                progress.set_postfix(step=nominal_step, task=task['task_id'], skip='incomplete', trained=optimizer_applied_steps)
                if should_save_checkpoint(step, total_steps):
                    save_checkpoint(run_dir, nominal_step, model, optimizer, optimizer_applied_steps)
                continue
            reward_values = [float(sample['reward']) for sample in samples]
            optimizer_applied, skip_reason = decide_training_step(rewards=reward_values, group_complete=True)
            mean_length = fmean(sample['completion_tokens'] for sample in samples)
            if not optimizer_applied:
                skip_counts[skip_reason or 'unknown'] += 1
                metric = build_training_metric(
                    nominal_step=nominal_step, task=task_row, rewards=reward_values,
                    kl=None, entropy=None, mean_completion_length=mean_length,
                    optimizer_applied=False, skip_reason=skip_reason, group_rewards=reward_values,
                )
                append_jsonl(metrics_path, metric)
                for sample in samples:
                    append_jsonl(rollout_path, {k: v for k, v in sample.items() if k not in ('prompt_ids', 'completion_ids')})
                live.log_training_step(metric, beta=beta)
                progress.set_postfix(step=nominal_step, task=task['task_id'], skip=skip_reason, trained=optimizer_applied_steps)
                if should_save_checkpoint(step, total_steps):
                    save_checkpoint(run_dir, nominal_step, model, optimizer, optimizer_applied_steps)
                continue

            rewards = torch.tensor(reward_values, device='cuda')
            reward_std = rewards.std()
            advantages = (rewards - rewards.mean()) / (reward_std + 1e-4)
            optimizer.zero_grad()
            policy_logps, kls, entropies = [], [], []
            for sample in samples:
                policy_logp, k3, entropy = token_statistics(model, sample)
                policy_logps.append(policy_logp)
                kls.append(k3)
                entropies.append(entropy)
            policy_logps = torch.stack(policy_logps)
            kls = torch.stack(kls)
            loss = -(advantages.detach() * policy_logps).mean() + beta * kls.mean()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer_applied_steps += 1

            metric = build_training_metric(
                nominal_step=nominal_step, task=task_row, rewards=reward_values,
                kl=kls.mean().item(), entropy=torch.stack(entropies).mean().item(),
                mean_completion_length=mean_length, optimizer_applied=True, skip_reason=None,
                group_rewards=reward_values,
            )
            metric['loss'] = loss.item()
            append_jsonl(metrics_path, metric)
            mlflow.log_metrics({k: v for k, v in metric.items() if isinstance(v, (int, float))}, step=nominal_step)
            for sample in samples:
                append_jsonl(rollout_path, {k: v for k, v in sample.items() if k not in ('prompt_ids', 'completion_ids')})
            live.log_training_step(metric, beta=beta)
            progress.set_postfix(
                step=nominal_step,
                task=task['task_id'],
                reward=f'{metric["mean_reward"]:.2f}',
                kl=f'{metric["kl"]:.3f}',
                trained=optimizer_applied_steps,
            )
            if should_save_checkpoint(step, total_steps):
                save_checkpoint(run_dir, nominal_step, model, optimizer, optimizer_applied_steps)
        progress.close()
        live.emit(f'{run_name} done: optimizer_applied={optimizer_applied_steps}, skip_counts={dict(skip_counts)}')
        live.refresh_panel()
        print(f'{run_name}: optimizer_applied={optimizer_applied_steps}, skip_counts={dict(skip_counts)}')
    model.save_pretrained(run_dir / 'final-adapter')
    tokenizer.save_pretrained(run_dir / 'final-adapter')
    return model, tokenizer, run_dir


preflight_model, preflight_tokenizer = build_policy()
PREFLIGHT = run_preflight(preflight_model, preflight_tokenizer)
TRAINABLE_IDS = set(PREFLIGHT['trainable_task_ids'])
TRAINING_SCHEDULE = build_training_schedule(TRAINABLE_IDS, TRAIN_TASK_ROWS, TRAIN_STEPS)
(OUTPUT_ROOT / 'preflight.json').write_text(json.dumps(PREFLIGHT, indent=2))
print('Adaptive train pool:', PREFLIGHT['trainable_by_tier'], 'count=', len(TRAINABLE_IDS))
del preflight_model
torch.cuda.empty_cache()

smoke_model, smoke_tokenizer, smoke_dir = train_run(1e-3, 'smoke', SMOKE_STEPS, TRAINABLE_IDS, TRAINING_SCHEDULE[:SMOKE_STEPS])
del smoke_model
torch.cuda.empty_cache()
smoke_checkpoints = sorted(smoke_dir.glob('checkpoint-*'), key=lambda p: int(p.name.split('-')[1]))
assert smoke_checkpoints, 'Smoke produced no checkpoint'
smoke_model, smoke_tokenizer, smoke_dir = train_run(
    1e-3, 'smoke', SMOKE_STEPS + 1, TRAINABLE_IDS, TRAINING_SCHEDULE[:SMOKE_STEPS + 1], smoke_checkpoints[-1],
)
smoke_metrics = read_jsonl(smoke_dir / 'metrics.jsonl')
assert sum(row.get('optimizer_applied') for row in smoke_metrics) >= 1
del smoke_model
torch.cuda.empty_cache()
print('Smoke and resume passed')

In [ ]:
low_model, low_tokenizer, low_dir = train_run(1e-5, 'beta-1e-5', TRAIN_STEPS, TRAINABLE_IDS, TRAINING_SCHEDULE)
LOW_HELDOUT = evaluate_policy(low_model, low_tokenizer, 'beta-1e-5')
del low_model
torch.cuda.empty_cache()

strong_model, strong_tokenizer, strong_dir = train_run(1e-2, 'beta-1e-2', TRAIN_STEPS, TRAINABLE_IDS, TRAINING_SCHEDULE)
STRONG_HELDOUT = evaluate_policy(strong_model, strong_tokenizer, 'beta-1e-2')
del strong_model
torch.cuda.empty_cache()

COMPARISON = {'base': BASE_HELDOUT, 'beta-1e-5': LOW_HELDOUT, 'beta-1e-2': STRONG_HELDOUT}
(OUTPUT_ROOT / 'heldout_comparison.json').write_text(json.dumps(COMPARISON, indent=2))
print(json.dumps(COMPARISON, indent=2))

In [ ]:
import matplotlib.pyplot as plt

FIGURE_DIR = OUTPUT_ROOT / 'figures'
FIGURE_DIR.mkdir(exist_ok=True)
runs = {
    'beta=1e-5': read_jsonl(low_dir / 'metrics.jsonl'),
    'beta=1e-2': read_jsonl(strong_dir / 'metrics.jsonl'),
}


def save_curve(field, ylabel, filename):
    figure, axis = plt.subplots(figsize=(7, 4))
    for label, rows in runs.items():
        points = [(row['nominal_step'], row[field]) for row in rows if row.get(field) is not None and row.get('optimizer_applied')]
        axis.plot([x for x, _ in points], [y for _, y in points], marker='o', markersize=3, label=label)
    axis.set(xlabel='Nominal step', ylabel=ylabel, title=ylabel)
    axis.grid(alpha=0.25)
    axis.legend()
    figure.savefig(FIGURE_DIR / f'{filename}.png', bbox_inches='tight')
    plt.show()


save_curve('mean_reward', 'Mean Phase 04 reward', 'reward')
save_curve('kl', 'Frozen-reference k3 KL', 'kl')
save_curve('entropy', 'Policy token entropy', 'entropy')
save_curve('mean_completion_length', 'Mean completion length', 'completion-length')

figure, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for tier, axis in zip((1, 2, 3), axes):
    field = f'tier_{tier}_pass_rate'
    for label, rows in runs.items():
        points = [(row['nominal_step'], row[field]) for row in rows if row.get(field) is not None and row.get('optimizer_applied')]
        axis.plot([x for x, _ in points], [y for _, y in points], marker='o', label=label)
    axis.set(title=f'Tier {tier}', xlabel='Nominal step')
    axis.grid(alpha=0.25)
axes[0].set_ylabel('Exact pass rate')
axes[-1].legend()
figure.tight_layout()
figure.savefig(FIGURE_DIR / 'per-tier-pass-rate.png', bbox_inches='tight')
plt.show()

labels = list(COMPARISON)
means = [COMPARISON[label]['mean_reward_scored'] or 0 for label in labels]
figure, axis = plt.subplots(figsize=(7, 4))
axis.bar(labels, means)
axis.set(title='Held-out (single rollout / task)', ylabel='Mean scored reward', ylim=(0, 1))
figure.savefig(FIGURE_DIR / 'heldout-before-after.png', bbox_inches='tight')
plt.show()

AUDIT = {'beta-1e-5': exploit_search(low_dir), 'beta-1e-2': exploit_search(strong_dir)}
(OUTPUT_ROOT / 'exploit_search.json').write_text(json.dumps(AUDIT, indent=2))
with mlflow.start_run(run_name='phase07-report'):
    mlflow.log_artifacts(str(FIGURE_DIR), artifact_path='figures')
    mlflow.log_artifact(str(OUTPUT_ROOT / 'heldout_comparison.json'))
    mlflow.log_artifact(str(OUTPUT_ROOT / 'exploit_search.json'))
    mlflow.log_artifact(str(OUTPUT_ROOT / 'preflight.json'))
print(json.dumps(AUDIT, indent=2))
print('Manually review candidate transcripts before writing conclusions.')